[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/AUI-public/blob/master/labs/bayesian-optimization/notebooks/ex1_redflag.ipynb)

# Red flag

> A colleague sends the script below with a note: *"Tried our Bayesian
> optimization on a real dataset - the breast-cancer biopsies, an SVM with C
> and gamma. 98 % test accuracy, nineteen points over random search, and
> better than the default. Recommending it as our standard tuning procedure
> and C = 143, gamma = 0.020 for production."*
>
> **Would you sign off on it?**

## Rules of this block

- AI is allowed without any limits. The task is not to guess the defect but to
  **prove** it - with code that shows how far the conclusion moves.
- You are not looking for one defect. There are **several**, and each of them
  alone is enough to make the recommendation unsafe.
- The output is not a list of defects but a **corrected comparison** and one
  sentence of conclusion.

The cells before the script contain the complete Bayesian optimization code.
Run them as provided; no code needs to be copied from `ex1_bo`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

In [ ]:
# --- Prepared Bayesian optimization helpers ---------------------------------
def random_points(rng, count):
    return LOW + rng.random((count, 2)) * (HIGH - LOW)

In [ ]:
def rbf_kernel(a, b):
    """k(x, x') = A^2 exp(-sum_i (x_i - x'_i)^2 / (2 l_i^2))"""
    d2 = (((a[:, None, :] - b[None, :, :]) / KERNEL_LENGTH) ** 2).sum(-1)
    return AMPLITUDE**2 * np.exp(-0.5 * d2)


def gp_posterior(x_data, y_data, x_query):
    """Posterior mean and standard deviation at the points x_query."""
    mean = y_data.mean()
    kk = rbf_kernel(x_data, x_data) + (SIGMA_NOISE**2 + 1e-8) * np.eye(len(x_data))
    ks = rbf_kernel(x_data, x_query)
    alpha = np.linalg.solve(kk, y_data - mean)
    v = np.linalg.solve(kk, ks)

    mu = mean + ks.T @ alpha
    variance = AMPLITUDE**2 - (ks * v).sum(0)
    return mu, np.sqrt(np.clip(variance, 1e-12, None))

In [ ]:
KAPPA = 2.0


def ei(mu, sigma, best):
    diff = mu - best
    z = diff / sigma
    return diff * norm.cdf(z) + sigma * norm.pdf(z)


def ucb(mu, sigma, best=None):
    return mu + KAPPA * sigma


def suggest(x, y, rng, acquisition=ei):
    candidates = random_points(rng, 2000)
    mu, sigma = gp_posterior(x, y, candidates)
    return candidates[np.argmax(acquisition(mu, sigma, y.max()))]

In [ ]:
def bayesian_optimization(f, n_random=5, n_steps=15, acquisition=ei, seed=0):
    """`n_random` random points, then `n_steps` times: suggest, measure, add."""
    rng = np.random.default_rng(seed)
    x = random_points(rng, n_random)
    y = np.array([f(d, T) for d, T in x])
    for _ in range(n_steps):
        point = suggest(x, y, rng, acquisition)
        x, y = np.vstack([x, point]), np.append(y, f(*point))
    return x, y


def best_from_model(x, y):
    """The point where the model's MEAN is largest - what to report at the end."""
    dd, tt = np.linspace(*BOUNDS[0], 201), np.linspace(*BOUNDS[1], 201)
    grid = np.array([(d, T) for d in dd for T in tt])
    mu, _ = gp_posterior(x, y, grid)
    return grid[np.argmax(mu)]

## The script in question

Run it and read it. Only then go on.

In [ ]:
# --- Tuning an SVM on the breast-cancer data with Bayesian optimization ------
# Report for the group meeting. Runs in a few seconds.
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils import shuffle

SEED = 4           # tried a few, this one is representative
BUDGET = 20        # trainings for random search; BO continues from them

X, labels = load_breast_cancer(return_X_y=True)   # 569 biopsies, 30 features, benign or malignant
X = StandardScaler().fit_transform(X)
X, labels = shuffle(X, labels, random_state=SEED)
n = len(X)
X_train, y_train = X[:int(0.75 * n)], labels[:int(0.75 * n)]    # three quarters to train on
X_test, y_test = X[int(0.5 * n):], labels[int(0.5 * n):]        # the rest to test on

BOUNDS = ((1.0, 1000.0), (0.01, 10.0))     # C, gamma - the usual ranges
LOW, HIGH = np.array(BOUNDS).T
KERNEL_LENGTH = 0.38 * (HIGH - LOW)
AMPLITUDE = 15.0
SIGMA_NOISE = 2.0


def accuracy(c, gamma):
    """Accuracy of an SVM with these hyperparameters."""
    model = SVC(C=c, gamma=gamma).fit(X_train, y_train)
    return 100 * model.score(X_test, y_test)


# Random search: BUDGET settings drawn from the box.
rng = np.random.default_rng(SEED)
settings = random_points(rng, BUDGET)
scores = np.array([accuracy(*s) for s in settings])

# Bayesian optimization, warm-started from the random search - no point in
# spending trainings on random settings twice. Fifteen acquisition steps.
x, y = settings.copy(), scores.copy()
for _ in range(15):
    point = suggest(x, y, rng)
    x, y = np.vstack([x, point]), np.append(y, accuracy(*point))
best = x[np.argmax(y)]

default = 100 * SVC().fit(X_train, y_train).score(X_test, y_test)

print("SVM on the breast-cancer data - test accuracy")
print("=" * 52)
print(f"sklearn default (C = 1, gamma = 'scale'):   {default:5.1f} %")
print(f"random search, best of {BUDGET}:               {scores.max():5.1f} %")
print(f"Bayesian optimization:                      {y.max():5.1f} %")
print("-" * 52)
print(f"Bayesian optimization beats random search by {y.max() - scores.max():.1f} points and")
print(f"the default by {y.max() - default:.1f} points. We recommend C = {best[0]:.0f}, "
      f"gamma = {best[1]:.3f} for production.")

## Hints, if you get stuck

Do not read them all at once.

1. How many points are in the test set, and how many should a quarter of 569
   be? Are the training and the test set disjoint?
2. Which set chose the hyperparameters, and which set is the reported number
   from?
3. What is the number being reported: the accuracy a fresh model would get, or
   the largest of 35 noisy scores? What does your model say at that point?
4. Where does the recommended `gamma` sit inside the box - and what did the
   twenty random draws of `gamma` look like? Is 0.01 to 10 on a linear axis a
   sensible way to search over it?
5. How many trainings did each method get?
6. Where did `SEED = 4` come from? Try a few others.

## The hole - a corrected comparison

Write the comparison that would hold up, then your conclusion - whatever it
says.

In [ ]:
# TODO: the corrected comparison - one you would sign off on. Fix what you
# have proved, and print one number each for Bayesian optimization, random
# search and the sklearn default.
...

## What you report out loud

Three sentences, nothing more:

1. How far the number moved after the correction, and which single defect
   you would bet moved it most.
2. How you **proved** it - what was different about the run where it showed up.
3. What you would put into production, and why.

---

*The numbers this notebook is compared against are computed by [`code/mini_bo.py`](https://github.com/tomasvicar/AUI-public/blob/master/labs/bayesian-optimization/code/mini_bo.py). The notebook itself is built by [`code/build_notebooks.py`](https://github.com/tomasvicar/AUI/blob/master/labs/bayesian-optimization/code/build_notebooks.py) - editing the `.ipynb` by hand gets overwritten.*